In [ ]:
pip install -U FlagEmbedding

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import json
from sentence_transformers import SentenceTransformer
import chromadb

# ==========================================
# 1. ĐỌC DỮ LIỆU ĐÃ BĂM (CHUNKING)
# ==========================================
print("1. Đang đọc file JSON...")
with open('CHUNKED_RECURSIVE_DATA.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Tùy thuộc vào code của bạn Long, data có thể là list các chuỗi, hoặc list các dictionary.
# Đoạn code này sẽ trích xuất phần text ra thành một list chuẩn:
chunks = []
metadatas = []

for item in data:

    if isinstance(item, dict):

        text_content = item.get(
            'page_content',
            item.get('text', '')
        )

        metadata = item.get('metadata', {})

        if text_content:
            chunks.append(text_content)
            metadatas.append(metadata)
print(f"-> Đã tìm thấy {len(chunks)} đoạn văn bản (chunks).")

# ==========================================
# 2. XỬ LÝ TIỀN TỐ CHO MODEL E5
# ==========================================
# Đây là "tử huyệt" của model E5: Bắt buộc phải thêm chữ "passage: " trước tài liệu
formatted_chunks = ["passage: " + chunk for chunk in chunks]

# ==========================================
# 3. TẢI MÔ HÌNH EMBEDDING
# ==========================================
print("\n2. Đang tải mô hình multilingual-e5-base...")
# Lưu ý: Lần đầu chạy nó sẽ tải khoảng 1.1GB model về máy, các lần sau sẽ rất nhanh
model = SentenceTransformer("intfloat/multilingual-e5-base")
# ==========================================
# 4. KHỞI TẠO VECTOR DATABASE
# ==========================================
print("\n3. Đang khởi tạo ChromaDB...")
# Dòng này sẽ tạo một thư mục tên 'db_khoa_cntt' ngay trong project của bạn để lưu file
chroma_client = chromadb.PersistentClient(path="./db_khoa_cntt")

# Xóa collection cũ nếu chạy lại nhiều lần để tránh trùng lặp
try:
    chroma_client.delete_collection("tai_lieu_khoa_cntt")
except:
    pass

collection = chroma_client.create_collection(
    name="tai_lieu_khoa_cntt",
    metadata={"hnsw:space": "cosine"}
)

# ==========================================
# 5. ÉP VECTOR VÀ LƯU VÀO KHO (BATCH PROCESSING)
# ==========================================
print("\n4. Đang ép Vector và lưu vào DB (Quá trình này tốn chút thời gian tùy RAM/GPU)...")

# Chia nhỏ ra ép từng đợt (batch) 32 đoạn một lúc để không bị tràn RAM máy tính
batch_size = 32 
for i in range(0, len(chunks), batch_size):
    batch_chunks = chunks[i:i+batch_size]
    batch_formatted = formatted_chunks[i:i+batch_size]
    
    embeddings = model.encode(
    batch_formatted,
    normalize_embeddings=True
    ).tolist()
    
    # Tạo ID duy nhất cho mỗi đoạn
    ids = [f"chunk_{j}" for j in range(i, i + len(batch_chunks))]
    
    # Lưu vào ChromaDB
    collection.add(
        documents=batch_chunks,  # Lưu text gốc (KHÔNG có chữ passage:) để mốt lôi ra cho LLM đọc
        embeddings=embeddings,
        metadatas=metadatas[i:i+len(batch_chunks)],   # Lưu mảng số để so sánh
        ids=ids
    )
    print(f" -> Đã xử lý và lưu: {min(i + batch_size, len(chunks))} / {len(chunks)} chunks.")

print("\n🎉 HOÀN TẤT! Toàn bộ Vector đã được lưu an toàn tại thư mục ./db_khoa_cntt")

1. Đang đọc file JSON...
-> Đã tìm thấy 561 đoạn văn bản (chunks).

2. Đang tải mô hình multilingual-e5-base...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


3. Đang khởi tạo ChromaDB...

4. Đang ép Vector và lưu vào DB (Quá trình này tốn chút thời gian tùy RAM/GPU)...
 -> Đã xử lý và lưu: 32 / 561 chunks.
 -> Đã xử lý và lưu: 64 / 561 chunks.
 -> Đã xử lý và lưu: 96 / 561 chunks.
 -> Đã xử lý và lưu: 128 / 561 chunks.
 -> Đã xử lý và lưu: 160 / 561 chunks.
 -> Đã xử lý và lưu: 192 / 561 chunks.
 -> Đã xử lý và lưu: 224 / 561 chunks.
 -> Đã xử lý và lưu: 256 / 561 chunks.
 -> Đã xử lý và lưu: 288 / 561 chunks.
 -> Đã xử lý và lưu: 320 / 561 chunks.
 -> Đã xử lý và lưu: 352 / 561 chunks.
 -> Đã xử lý và lưu: 384 / 561 chunks.
 -> Đã xử lý và lưu: 416 / 561 chunks.
 -> Đã xử lý và lưu: 448 / 561 chunks.
 -> Đã xử lý và lưu: 480 / 561 chunks.
 -> Đã xử lý và lưu: 512 / 561 chunks.
 -> Đã xử lý và lưu: 544 / 561 chunks.
 -> Đã xử lý và lưu: 561 / 561 chunks.

🎉 HOÀN TẤT! Toàn bộ Vector đã được lưu an toàn tại thư mục ./db_khoa_cntt
